# 07 — Makinat me vektorë mbështetës (SVM)

        **Pyetja udhëheqëse:** Si gjejmë jo vetëm një kufi ndarës, por kufirin me margjinën më të madhe?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të kuptohet hiperplani dhe margjina.
- Të interpretohen vektorët mbështetës.
- Të studiohet parametri C.
- Të vizualizohet efekti i kernelit RBF.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("07 — Makinat me vektorë mbështetës (SVM)")
print("Seed:", RANDOM_STATE)

# 1. Gjeometria e ndarjes — CORE

In [ ]:
from sklearn.datasets import make_blobs
X,y=make_blobs(n_samples=240,centers=[(-1.6,-1.1),(1.4,1.2)],cluster_std=[0.75,0.8],random_state=RANDOM_STATE)
fig,ax=plt.subplots()
ax.scatter(X[:,0],X[:,1],c=y,s=18)
ax.set_title("Dy klasa pothuajse linearisht të ndashme")
plt.show()

Një hiperplan shkruhet
$$
\mathbf w^T\mathbf x+b=0.
$$
Distanca e një pike nga hiperplani është
$$
d=\frac{|\mathbf w^T\mathbf x+b|}{\|\mathbf w\|}.
$$
SVM zgjedh kufirin që maksimizon margjinën, duke u mbështetur veçanërisht te pikat më afër kufirit:
**vektorët mbështetës (support vectors)**.

# 2. SVM linear dhe margjina — CORE

In [ ]:
from sklearn.svm import SVC
clf=SVC(kernel="linear",C=1.0).fit(X,y)
w=clf.coef_[0]; b=clf.intercept_[0]
xx=np.linspace(X[:,0].min()-1,X[:,0].max()+1,300)
yy=-(w[0]*xx+b)/w[1]
margin=1/np.linalg.norm(w)
yy1=-(w[0]*xx+b-1)/w[1]
yy2=-(w[0]*xx+b+1)/w[1]

fig,ax=plt.subplots()
ax.scatter(X[:,0],X[:,1],c=y,s=16)
ax.plot(xx,yy)
ax.plot(xx,yy1,linestyle="--"); ax.plot(xx,yy2,linestyle="--")
ax.scatter(clf.support_vectors_[:,0],clf.support_vectors_[:,1],s=90,facecolors="none",edgecolors="black")
ax.set_title(f"SVM linear; margjina gjeometrike ~ {margin:.2f}")
plt.show()

# 3. Soft margin dhe parametri C — CORE

In [ ]:
Cs=[0.05,1,100]
fig,ax=plt.subplots()
for C in Cs:
    m=SVC(kernel="linear",C=C).fit(X,y)
    ax.scatter(C,len(m.support_),s=80)
ax.set_xscale("log")
ax.set_xlabel("C"); ax.set_ylabel("Numri i vektorëve mbështetës")
ax.set_title("C kontrollon kompromisin margjinë–gabime")
plt.show()

# 4. Hinge loss — CORE

In [ ]:
margins=np.linspace(-2,3,400)
hinge=np.maximum(0,1-margins)
fig,ax=plt.subplots()
ax.plot(margins,hinge)
ax.axvline(1,linestyle="--")
ax.set_xlabel("y f(x)"); ax.set_ylabel("hinge loss")
ax.set_title("Humbja hinge")
plt.show()

# 5. Kernel RBF për kufij jolinearë — CORE

In [ ]:
from sklearn.datasets import make_circles
Xc,yc=make_circles(n_samples=500,factor=0.45,noise=0.09,random_state=RANDOM_STATE)
lin=SVC(kernel="linear").fit(Xc,yc)
rbf=SVC(kernel="rbf",gamma="scale",C=2).fit(Xc,yc)

xx,yy=np.meshgrid(np.linspace(-1.5,1.5,250),np.linspace(-1.5,1.5,250))
for name,m in [("linear",lin),("RBF",rbf)]:
    zz=m.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
    fig,ax=plt.subplots()
    ax.contourf(xx,yy,zz,alpha=0.2)
    ax.scatter(Xc[:,0],Xc[:,1],c=yc,s=10)
    ax.set_title(f"SVM {name}")
    plt.show()

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Sensitivitet ndaj gamma — EXTENDED

In [ ]:
gammas=[0.2,1,8]
fig,ax=plt.subplots()
scores=[]
for g in gammas:
    m=SVC(kernel="rbf",gamma=g,C=2).fit(Xc,yc)
    scores.append(m.score(Xc,yc))
ax.plot(gammas,scores,marker="o")
ax.set_xscale("log"); ax.set_xlabel("gamma"); ax.set_ylabel("accuracy trajnim")
ax.set_title("Gamma dhe kompleksiteti i kufirit")
plt.show()

# 7. Aplikim jo-fizik: klasifikim tekstesh — EXTENDED

SVM-të lineare kanë qenë prej kohësh shumë efektive në hapësira me dimension të lartë, p.sh. tekste të përfaqësuara me mijëra veçori të rralla.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Llogaritni manualisht distancën e një pike nga hiperplani.
2. Analizoni si ndryshon numri i support vectors me C.
3. Krahasoni linear dhe RBF në `make_circles`.
4. Shpjegoni pse scaling është i rëndësishëm për SVM.

        ## B. Ushtrim i plotë
        Krijoni dy regjime dinamike me mbivendosje dhe përdorni grid-search për të zgjedhur C dhe gamma.

        ## C. Playground
        Ndryshoni C, gamma, zhurmën dhe standardizimin.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.